In [ ]:
!wget https://raw.githubusercontent.com/alexeygrigorev/minsearch/main/minsearch.py
git c

--2026-05-08 09:08:00--  https://raw.githubusercontent.com/alexeygrigorev/minsearch/main/minsearch.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4404 (4.3K) [text/plain]
Saving to: ‘minsearch.py.9’

minsearch.py.9      100%[===================>]   4.30K  --.-KB/s    in 0s      

2026-05-08 09:08:01 (17.5 MB/s) - ‘minsearch.py.9’ saved [4404/4404]



In [75]:
import minsearch

In [76]:
import json

In [77]:
#!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/tree/main/cohorts/2025/01-intro/documents.json
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2025/01-intro/documents.json

--2026-05-08 09:08:01--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2025/01-intro/documents.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 658332 (643K) [text/plain]
Saving to: ‘documents.json.9’

documents.json.9    100%[===================>] 642.90K  --.-KB/s    in 0.06s   

2026-05-08 09:08:02 (9.75 MB/s) - ‘documents.json.9’ saved [658332/658332]



In [78]:
with open('documents.json', 'rt') as f_in:
    docs_raw = json.load(f_in)


In [79]:
documents = []

for course_dict in docs_raw:
    for doc in course_dict['documents']:
        doc['course'] = course_dict['course']
        documents.append(doc)
        

In [80]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [81]:
index = minsearch.Index(
    text_fields=["question", "text", "section"],
    keyword_fields=["course"]
)

In [82]:
q = 'the course has already started, can I still enroll?'

In [83]:
q

'the course has already started, can I still enroll?'

In [84]:
index.fit(documents)

In [85]:
boost = {'question': 3.0, 'section': 0.5}

results = index.search(
    query=q,
    filter_dict={'course': 'data-engineering-zoomcamp'},
    boost_dict=boost,
    num_results=5
    
)

In [86]:
results

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 202

In [87]:
from openai import OpenAI
import os

In [91]:
q

'the course has already started, can I still enroll?'

In [92]:
response = client.chat.completions.create(
    model='gpt-4o',
    messages=[{"role":"user","content":q}]
)

In [93]:
response.choices[0].message.content

"Whether you can still enroll in a course that has already started depends on several factors, including the institution's policies, the course structure, and the reason for your late entry. Here are a few steps you can take:\n\n1. **Check the Enrollment Policy**: Look at the institution’s website or contact the registrar’s office to understand their policy on late enrollment.\n\n2. **Contact the Instructor**: Reach out to the course instructor or professor directly. They might allow late enrollment, especially if the course is not too far along.\n\n3. **Consider Course Type**: Some courses, especially online or self-paced ones, may have flexible start dates and allow late enrollments.\n\n4. **Understand the Implications**: Be aware that joining a course late can mean catching up on missed material, assignments, and possibly exams.\n\n5. **Ask About Fees**: Some institutions may charge a late enrollment fee.\n\n6. **Explore Alternatives**: If enrollment is closed, ask if there are othe

In [94]:
prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT. 
Use only the facts from the CONTEXT when answering the QUESTION.
if the CONTEXT doesn't contain the answer, output NONE


QUESTION: {question}

CONTEXT: 
{context}
""".strip()

In [95]:
context = ""

for doc in results:
    context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"

In [96]:
prompt = prompt_template.format(question=q, context=context).strip()

In [97]:
print(prompt)

You're a course teaching assistant. Answer the QUESTION based on the CONTEXT. 
Use only the facts from the CONTEXT when answering the QUESTION.
if the CONTEXT doesn't contain the answer, output NONE


QUESTION: the course has already started, can I still enroll?

CONTEXT: 
section: General course-related questions
question: Course - Can I still join the course after the start date?
answer: Yes, even if you don't register, you're still eligible to submit the homeworks.
Be aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.

section: General course-related questions
question: Course - Can I follow the course after it finishes?
answer: Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.
You can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project

In [98]:
response = client.chat.completions.create(
    model='gpt-4o',
    messages=[{"role":"user","content":prompt}]
)

response.choices[0].message.content

'Yes, you can still enroll in the course after it has started. You are eligible to submit the homework, but be mindful of the deadlines for turning in the final projects.'

In [99]:
def search(query):
    boost = {'question': 3.0, 'section': 0.5}

    results = index.search(
        query=query,
        filter_dict={'course': 'data-engineering-zoomcamp'},
        boost_dict=boost,
        num_results=10
        
    )

    return results

In [100]:
query = 'how do I run kafka?'
search_results = search(query)

In [101]:
def build_prompt(query, search_results):
    prompt_template = """
    You're a course teaching assistant. Answer the QUESTION based on the CONTEXT. 
    Use only the facts from the CONTEXT when answering the QUESTION.    
    
    QUESTION: {question}
    
    CONTEXT: 
    {context}
    """.strip()

    context = ""
    
    for doc in search_results:
        context = context + f"section: {doc['section']}\nquestion: {doc['question']}\nanswer: {doc['text']}\n\n"

    prompt = prompt_template.format(question=q, context=context).strip()
    return prompt

In [102]:
def llm(prompt):
    response = client.chat.completions.create(
    model='gpt-4o',
    messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content

In [103]:
#query = 'the course has already started, can i still enroll'
query = 'how do I run kafka'

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [104]:
rag('the course has just started, can i still enroll')

"Yes, you can still enroll in the course even after it has started. You are eligible to submit the homeworks as well. However, be mindful of the deadlines for turning in the final projects, so don't leave everything for the last minute."

In [105]:
documents[0]

{'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
 'section': 'General course-related questions',
 'question': 'Course - When will the course start?',
 'course': 'data-engineering-zoomcamp'}

In [106]:
from elasticsearch import Elasticsearch

In [107]:
es_client = Elasticsearch('http://localhost:9200')

In [108]:
es_client.info()

ObjectApiResponse({'name': '5a1202547551', 'cluster_name': 'docker-cluster', 'cluster_uuid': '5VCV3kpeQeWM3bSIivl2zg', 'version': {'number': '9.0.0', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '112859b85d50de2a7e63f73c8fc70b99eea24291', 'build_date': '2025-04-08T15:13:46.049795831Z', 'build_snapshot': False, 'lucene_version': '10.1.0', 'minimum_wire_compatibility_version': '8.18.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'})

In [110]:
#Create an index is elastic search
index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            "text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"}
        }
    }
}

index_name = "course-questions"

es_client.indices.delete(index=index_name, ignore_unavailable=True)

es_client.indices.create(index=index_name, body=index_settings)
            
    

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [111]:
from tqdm.auto import tqdm

In [112]:
#Document the index
for doc in tqdm(documents):
    es_client.index(index=index_name, document=doc)

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 948/948 [00:03<00:00, 265.50it/s]


In [113]:
query = 'I just discovered the course. Can I still join it?'

In [114]:
def elastic_search(query):
    search_query = {
        "size": 5,
        "query": {
            "bool": {
                "must": {
                    "multi_match": {
                        "query": query,
                        "fields": ["question^3", "text", "section"],
                        "type": "best_fields"
                    }
                },
                "filter": {
                    "term": {
                        "course": "data-engineering-zoomcamp"
                    }
                }
            }
        }
    }

    response = es_client.search(index=index_name, body=search_query)

    result_docs = []
    for hit in response['hits']['hits']:
        result_docs.append(hit['_source'])

    return result_docs

In [115]:
elastic_search(query)

[{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account\nGoogle Cloud SDK\nPython 3 (insta

In [119]:
def rag(query):
    search_results = elastic_search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

In [124]:
rag(query)

'Yes, you can still enroll in the course even after it has started. You are eligible to submit the homework, but keep in mind that there will be deadlines for turning in the final projects, so try not to leave everything for the last minute.'